In [ ]:
# 0: ancillary qubit for block encoding
# 1: wire_i for block encoding
# 2,3: 00 for psi_00, 01 for psi_01, 10 for psi_10
# 4: wire_j
# 5: Ancillary qubit for measurement

In [ ]:
from qiskit import QuantumCircuit, ClassicalRegister
from numpy import sqrt, array
from numpy.linalg import norm


def circuit_init():
    qc = QuantumCircuit(6)
    return qc

def PREP(qc):
    desired_vector = [sqrt(1/3), 0, sqrt(2/9), 1/3, sqrt(2/9), -1/3, 0, 0]
    qc.initialize(desired_vector, [4,3,2])
    return qc

In [ ]:
from pennylane.templates.state_preparations.mottonen import compute_theta, gray_code
import numpy as np
from qiskit import transpile
from qiskit_aer import Aer, AerSimulator

A = np.array ([
    [3/sqrt(7), 0],
    [0, 3/sqrt(2)]
])

ancilla_wire = [0]
wires_i = [1]
wires_j = [4]


s = int(np.log2(A.shape[0]))

A = A / np.max(np.abs(A))
thetas = np.arccos(A).flatten()
thetas = compute_theta(thetas)

code = gray_code(2 * np.log2(len(A)))
n_selections = len(code)
control_order = [int(np.log2(int(code[i], 2) ^ int(code[(i + 1) % n_selections], 2))) for i in range(n_selections)]

tolerance = 0.01
def UT(qc, thetas, control_wires, ancilla_wire):
    ancilla = ancilla_wire[0]  
    nots = []

    for theta, control_index in zip(thetas, control_wires):
        if abs(2 * theta) > tolerance:
            for c_wire in nots:
                qc.cx(c_wire, ancilla)
            qc.ry(2 * theta, ancilla)
            nots = []

        if control_index in nots:
            nots.remove(control_index)  
        else:
            nots.append(control_index)  

    for c_wire in nots:
        qc.cx(c_wire, ancilla)
        
def UB(qc, wires_i, wires_j):
    for w_i, w_j in zip(wires_i, wires_j):

        qc.swap(w_i, w_j)

def HN(qc, input_wires):
    for w in input_wires:
        qc.h(w)
        
input_wires = ancilla_wire + wires_i

def get_control_qubit(control_order, wires_i, wires_j):
    n = len(wires_i)  # Same as len(wires_j)
    control_qubits = []

    for control_value in control_order:
        # Control values between 0 and n-1 correspond to wires_j (reverse order)
        if control_value < n:
            control_qubits.append(wires_j[-(control_value + 1)])  # Reverse index for wires_j
        # Control values between n and 2n-1 correspond to wires_i (reverse order)
        elif control_value < 2 * n:
            control_qubits.append(wires_i[-(control_value - n + 1)])  # Reverse index for wires_i

    return control_qubits

control_wires = get_control_qubit(control_order, wires_i, wires_j)

def Block_Encoding(qc):
    HN(qc, wires_i)
    qc.barrier()  
    UT(qc, thetas, control_wires, ancilla_wire)
    qc.barrier()   
    UB(qc, wires_i, wires_j)
    HN(qc, wires_i)
    qc.barrier()
    
    return qc

In [ ]:
from qiskit.circuit import ParameterVector

def Unitary(qc, wires, num_layers):

    x = ParameterVector('x', 2 * num_layers)
    z = ParameterVector('z', 2 * num_layers)

    for i in range(num_layers):
        qc.barrier()
        qc.rx(x[2*i], wires[0])
        qc.rx(x[2*i+1], wires[1])

        qc.rz(z[2*i], wires[0])
        qc.rz(z[2*i+1], wires[1])
        
        # Apply CNOT gates between the qubits
        qc.cx(wires[0], wires[1])

    qc.measure_all()
    return qc

In [ ]:
from numpy.random import rand
num_layers=3
num_params= 4*num_layers
num_shots=1000

qc = circuit_init()
qc = PREP(qc)
qc.barrier()
qc = Block_Encoding(qc)
Unitary(qc, [wires_j[0],[5]] , num_layers)

qc_reversed=qc.reverse_bits()
qc.draw(output='mpl', fold=False, style = 'clifford') 

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService(channel='ibm_quantum')
#backend = service.least_busy(min_num_qubits=127)
backend = service.backend("ibm_strasbourg")
print(backend)

pm = generate_preset_pass_manager(optimization_level=3,backend=backend)

candidate_circuit = pm.run(qc_reversed)
candidate_circuit.draw('mpl', fold=False, scale =0.1, idle_wires=False)

In [ ]:
psi00 = '00'
psi01 = '01'
psi10 = '10'
psi = psi00  # Change this to psi01 or psi10 as needed


def cost_func_sampler(params, ansatz, sampler):

    target_counts = 0
    total_counts = 0
    
    # Run the estimator for all jobs
    pub = (ansatz, params)
    job = sampler.run([pub])
    
    results = job.result()[0]
    counts = results.data.meas.get_counts()

    for outcome, count in counts.items():
        if outcome[:4] == '00' + psi:  # Check b0b1b2b3: '00 00' for psi00, '00 01' for psi01, and '00 10' for psi10
            total_counts += count
            if outcome[4:6] == psi:  # Check b4b5: '00' for psi00, '01' for psi01, and '10' for psi10
                target_counts += count
    cost = target_counts/total_counts
    print(f"tot counts prob: {total_counts/num_shots}, tar counts prob: {target_counts/num_shots}, cost: {cost}")

    # Append cost to the global success probability list and return it
    confidences.append(cost)
    return 1 / cost

In [ ]:
from qiskit_ibm_runtime import Session, SamplerV2 as Sampler
from scipy.optimize import minimize

init_params = rand(num_params)
confidences = [] # Global variable

with Session(backend=backend) as session:

    sampler = Sampler(mode=session)
    sampler.options.default_shots = num_shots

    result = minimize(
        cost_func_sampler,
        init_params,
        args=(candidate_circuit, sampler),
        method="COBYLA",
        tol=1e-1,
    )
    print(result)
    print(result.x)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(confidences, label="Confidence")
plt.xlabel('Iteration')
plt.ylabel('Confidence')
plt.legend()
plt.show()

In [ ]:
import openpyxl

# create a new workbook
workbook = openpyxl.Workbook()

# select the active worksheet
worksheet = workbook.active

# loop through the confidences array and write values to the worksheet
for i in range(len(confidences)):
    worksheet.cell(row=i+1, column=1, value = float(confidences[i]))

# save the workbook to a file
workbook.save('MCM BE HEA Sym states00 (ibm_strasbourg).xlsx')

In [ ]:
from numpy import max
max(confidences)